In [1]:
# Вначале необходимо установить используемые библиотеки.

In [2]:
!pip install --upgrade langchain langchain-huggingface langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.1/113.1 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 97.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 61.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 543.9/543.9 kB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.8/173.8 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.9 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.3.1
    Uninstalling langchain-core-1.3.1:
      Successfully uninstalled langchain-core-1.3.1
  Attempting uninstall: langgraph-prebuilt
    Found existing installation: langgraph-prebuilt 1.0.10

In [3]:
!pip install langgraph

In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langchain_huggingface import HuggingFacePipeline
from langchain_huggingface import ChatHuggingFace
import torch

In [5]:
# Можно использовать разные модели. Чем больше размер модели, тем в среднем лучше результат.
# Также необходимо помнить что некоторые модели специально дообучаются для использования в качесвте агентов,
# их учат использовать созданные для них инструменты и следовать инструкциям.

# Указываем имя модели и загружаем токенизатор и модель
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    dtype=torch.float16,
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [6]:
# Создаем pipeline Hugging Face
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=1024,
    temperature=0.3,
    top_p=0.95,
    repetition_penalty=1.2,
    max_length=512
)

# На основе pipeline Hugging Face создаем langchain pipeline и langchain chat model
# И то и то можно использовать для построения агентной системы. Основное отличие — структура ввода и вывода.
# pipeline принимает и выдает текст, тогда как chat model работает с запросами в виде диалогов.
llm = HuggingFacePipeline(pipeline=pipe)
chat_model = ChatHuggingFace(llm=llm)

Passing `generation_config` together with generation-related arguments=({'repetition_penalty', 'max_new_tokens', 'max_length', 'top_p', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [12]:
datetime.now().strftime("%d-%m-%Y")

'06-05-2026'

In [14]:
from langchain.tools import tool
from datetime import datetime

@tool("get_current_date", description="Верни сегодняшнюю дату в формате DD-MM-YYYY")
def get_current_date() -> str:

    return datetime.now().strftime("%d-%m-%Y")

In [15]:
from langgraph.prebuilt import create_react_agent

tools = [get_current_date]
system_prompt = (
    f"Ты полезный ассистент. Если пользователь спросит какая сегодня дата, обязательно используй инструментом 'get_current_date' и не полагайся на свои мысли. Если ты воспользовался инструментом, то укажи это ."
)
agent_executor = create_react_agent(chat_model, tools,prompt=system_prompt)
input_message = {
    "role": "user",
    "content": "Какая сегодня дата?"
}

result = agent_executor.invoke({"messages": [input_message]})

for message in result["messages"]:
  message.pretty_print()


/tmp/ipykernel_1437/644514396.py:7: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_executor = create_react_agent(chat_model, tools,prompt=system_prompt)
Both `max_new_tokens` (=1024) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


================================ Human Message =================================

Какая сегодня дата?
================================== Ai Message ==================================

<|system|>
Ты полезный ассистент. Если пользователь спросит какая сегодня дата, обязательно используй инструментом 'get_current_date' и не полагайся на свои мысли. Если ты воспользовался инструментом, то укажи это .</s>
<|user|>
Какая сегодня дата?</s>
<|assistant|>
"Сегодня, как вы знаете, является четвёртой рабочим днем в месяце."

В данном случае "четвертая" (4) является числом, которое присутствует после запятой в строке, что соответствует суточному числе. В этом случае можно использовать инструмент get_current_date для получения текущей даты в формате YYYY-MM-DD или другого варианта, который будет определен в коде.


In [9]:
# Пользовательский запрос содержится в пользовательской части сообщения.
input_message = {
    "role": "user",
    "content": "Какая сегодня дата?"
}

result = agent_executor.invoke({"messages": [input_message]})
print(result["messages"][1])

Both `max_new_tokens` (=1024) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


content='<|system|>\nТы полезный ассистент. Если пользователь спросит какое сегодня число, обязательно используй \'get_current_date\' и не полагайся на свои мысли. если ты воспользовался агентом, то укажи это .</s>\n<|user|>\nКакая сегодня дата?</s>\n<|assistant|>\nДата сегодня - 12 сентября 2021 года. Вы можете использовать функцию `get_current_date` из библиотеки для получения данных о сегодняшнем дне:\n\n```python\nimport datetime as dt\n\ntoday = dt.datetime.now()\nprint(f"Today is {dt.strftime(today, \'%A\')} on the year of {dt.strftime(today, \'%Y\')}.")\n```\n\nВ этом коде `dt.datetime.now()` возвращает объект типа `datetime`, который содержит текущее время в формате UTC (Coordinated Universal Time). Внутри этой переменной есть объекты типов `year`, `month`, `day`, которые позволяют получить нужные вам значения. Чтобы получить информацию о сегодняшнем дне, вы должны обратиться к этим значениям через методы `strftime()`. Например, чтобы получить название суток в формате "Mon", "T